# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 16 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Jeffrey Otoo
**Student ID:** 14302028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [ ]:
import sys

print(sys.executable)

/Users/jeffreyotoo/lab-4-llm-decision-support-1/.venv/bin/python


In [3]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---


# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",  # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [28]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


#
# TODO: Call it once with a simple question and print the answer.
print(ask_llm("What is 2+2"))
# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 2+2?"},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.usage)

2 + 2 = 4
CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041610774, prompt_time=0.002198617, completion_time=0.011168131, total_time=0.013366748)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 1. System vs user roles. Persistent behaviour (role, constraints, output format) is set in the system role and is used for all calls. The user role is associated with the specific request. The assistant to a microfinance loan officer in Ghana is given the task to summarize this loan application.The user is a microfinance loan officer in Ghana, who is asked to summarize this loan application. The system prompt is written only once and is repeated for each of the six letters.

I stumbled on the amount of work it does; my first time using V2, I forgot to set the system prompt to a variable, and it automatically returned to the default, so V1 and V2 returned the same.

2. Tokens. A token is a bit of a word chunk – about 3/4 of an english word, so common words are one token, names and numbers are multiple. My first call was 48 prompt tokens and 9 completion tokens, totaling 57. Providers charge per token because compute isn't proportional to the number of requests, but rather to the length of the sequence: a 2000-word letter costs them more than a 4-word question, and a flat per-request fee would be costing short users money for long users.

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
QUESTION = (
    "Suggest ONE name for a savings product for market traders in Accra. "
    "Reply with the name only, no explanation."
)

for temp in [0.0, 1.2]:
    print(f"\nTemperature: {temp}")
    for i in range(5):
        answer = ask_llm(QUESTION, temperature=temp, max_tokens=20)
        print(f"[{i + 1}] {answer.strip()}")

# TODO: Print all 10 answers, grouped by temperature.
print("\nAll answers grouped by temperature:")


Temperature: 0.0
[1] MakolaSave
[2] MakolaSave
[3] MakolaSave
[4] MakolaSave
[5] MakolaSave

Temperature: 1.2
[1] Makola Save
[2] MakolaSave
[3] MakolaSave
[4] Makola Savings Vault
[5] Makola Save

All answers grouped by temperature:


**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** The five answers to the question at temperature 0.0 were very similar in content (roughly the same ideas, same sequence). They were significantly different at 1.2.

Surprisingly, temperature 0 was not a perfectly deterministic process. There are two differences between Runs [1] and [2]: Item 4 is changed from "MarketMoen" to "Market Booster" and Item 7 is completely changed. The temperature=0 option is not "take the most likely token" but rather "take the token that is most likely to be generated from the system's current state" – if a system is required to be auditable, it should store the output it produced from the system's current state, not assume it can regenerate it.

The condition of low temperature is suitable for the loan system. Each component should be as close as possible to a source document; differences between runs are errors not diversity. If the same letter resulted in a different amount being extracted on Tuesday as it did on Monday, the institution would not be able to audit its own decisions and an applicant would not be able to be provided with a stable explanation.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [8]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [12]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    user_prompt = SUMMARY_PROMPT_V1.format(letter_text=letter_text)
    summary = ask_llm(user_prompt, temperature=0.0)
    print(f"\nSummary for {letter_id}:\n{summary}\n")


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana. Your task is to summarize loan application letters in a factional and neutral manner.
Do not invent any details. Keep your summary to 3-4 sentences.

Rules:
- Include the applicant's name, requested loan amount, purpose of the loan, and proposed repayment plan.
- If the letter mentions monthly profit, include that in the summary.
- Neutral tone. No praise or criticism of the applicant. No invented details."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    user_prompt = SUMMARY_PROMPT_V2.format(letter_text=letter_text)
    summary = ask_llm(user_prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)
    print(f"\nSummary for {letter_id}:\n{summary}\n")

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.



Summary for L002:
Kwame Boateng, a commercial driver from Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He is seeking urgent assistance.


Summary for L006:
Here's a summary of Kofi's loan application:

* Loan amount: GHS 50,000
* Business ideas: 
  1. Car washing business
  2. Provision shop
  3. Importing phones from Dubai
* Applicant's details: 22 years old, no prior business experience, but claims to be "business-minded"
* Repayment plan: Pay back the loan in 1 year, expecting the businesses to be profitable by then
* Collateral: None, but Kofi claims to be "trustworthy"


Summary for L002:
Kwame Boateng is applying for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He intends to repay the loan, but a s

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 1. Problems V1 had that V2 fixed.

Inference laundering. V1 on L002: Kwame "is relying on his future earnings to repay the loan." The letter says "I can pay back whenever the money comes." V1 converted a non-answer into something that reads like a strategy. V2 instead reports that "a specific repayment plan is not provided." This is the most dangerous failure because it is a reframing, not a fabricated fact, so nothing about it looks wrong.

Silence about gaps. V2 on L006 states "The application does not provide information on the applicant's current financial situation or projected monthly profits." V1 never does this, so a missing figure is indistinguishable from a summarizer that chose not to mention it.

No consistent shape. V1 gave L006 a six-item bullet list including "Applicant's age: 22" and L002 a five-sentence paragraph — six letters, six formats, defeating the point of a scannable brief.

V2 is not perfectly clean either: it writes "Mr. Boateng," a title the letter never uses.

2. "No invented details." The failure mode is hallucination (confabulation): fluent text not grounded in the input. It matters here because the output is evidence in a credit decision about a real person. An invented guarantor or inflated profit figure does not look wrong — it looks like a fact, and an officer reading forty briefs a day cannot catch it without re-reading every original letter, which removes the reason for using the system at all.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [25]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

EXTRACT_SYSTEM = (
    "You extract structured data from loan application letters. You output JSON only."
)

EXTRACT_PROMPT = """Extract these fields from the loan application letter below.

Return ONLY a JSON object with EXACTLY these keys:
  "applicant_name": string
  "amount_ghs": number
  "purpose": string
  "monthly_profit_ghs": number or null
  "has_collateral_or_guarantor": boolean
  "repayment_months": number or null

Rules:
- If a field is not stated in the letter, use null. Do NOT guess or estimate.
- has_collateral_or_guarantor is true only if the letter names specific collateral,
  a guarantor, a pledged asset, or group/joint liability. Vague reassurance such as
  "I am trustworthy" is NOT collateral.
- Numbers must be plain digits: 8000, not "GHS 8,000".
- No markdown fences, no explanation, no text outside the JSON object.

Example letter:
\"\"\"Dear Sir, I am Adjoa Nyarko and I run a small bakery in Tema. I request GHS 6,000
to buy a second oven. My shop clears about GHS 700 each month. I will repay over
10 months. My brother has offered his motorbike as security.\"\"\"

Example output:
{{"applicant_name": "Adjoa Nyarko", "amount_ghs": 6000, "purpose": "buy a second oven",
"monthly_profit_ghs": 700, "has_collateral_or_guarantor": true, "repayment_months": 10}}

Now extract from this letter:
\"\"\"{letter_text}\"\"\""""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).


def extract_fields(letter_text, temperature=0):
    raw = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=EXTRACT_SYSTEM,
        temperature=temperature,
    )
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"WARNING: could not parse JSON ({e}). Raw output was:\n{raw}\n")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
EXTRACTED = {}
rows = []

for letter_id in LETTERS:
    result = extract_fields(LETTERS[letter_id])
    EXTRACTED[letter_id] = result  # raw: None stays None
    rows.append({"letter_id": letter_id, **(result or {})})

df = pd.DataFrame(rows).set_index("letter_id")
df


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 1. Why the few-shot example must be invented. Two reasons. If it were L001, the correct answer would already be in the prompt and my 4.1 accuracy would measure copying, not reading. And — as my 4.3 test showed — the invented example is what makes leakage detectable: fed an empty string, the only concrete letter in context was my invented one (Adjoa Nyarko, GHS 6,000), and the model returned nulls rather than copying it. Had the example been L001, a leaking model would have returned Akosua's details, which look exactly like correct output.

2. "Use null, do not guess." Every field is shaped like an answer, and a model asked for a number tends to produce one. Without the rule the pressure is to fill the slot from context — inferring a repayment period for L002 from "after the festive season," or a plausible profit for L006. With it, my extractor returned null for monthly_profit_ghs on L002, L005 and L006 and for repayment_months on L002, all correct. I also had to state that vague reassurance like "I am trustworthy" is not collateral, which is what L006 tests — and it correctly returned false.

3. Temperature 0 for extraction. Extraction has one correct answer already present in the source, so variation is error, not diversity — and reproducibility is itself a requirement for a system making credit decisions. Creative tasks invert this: there is no single correct product name, and temperature 0 in Part 1.2 collapsed five runs onto one answer, which for naming is a worse outcome.

My 4.2 result refined this: across ten runs on L004 at both temperatures, every number and boolean was identical and only purpose, the one free-text field, varied. Temperature bites where the prompt constrains least.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM = """You are a decision-support assistant to a microfinance loan officer in Ghana.

You do NOT make lending decisions. Loan decisions are made by human officers.
Never state or imply that a loan should be approved, rejected, granted, or declined.
Never assign a score, grade, or probability of approval.

Ground every point in what the letter actually says. Do not invent facts, figures,
or circumstances. If something important is unstated, that belongs under Missing
information, not under Strengths or Risks."""

BRIEF_PROMPT = """Loan application letter:
\"\"\"{letter_text}\"\"\"

Extracted data:
{extracted_json}

Produce a brief with exactly these four sections:

1. Strengths
   Bullet points. Only factors evidenced in the letter.

2. Risks / red flags
   Bullet points. Include vagueness, unverified claims, and reliance on hope
   or future events as risks in their own right.

3. Missing information
   Bullet points. What should the officer ask the applicant for?

4. Suggested next step
   ONE line. Choose from: "invite for interview", "request supporting documents",
   "conduct site visit", "flag for senior review", "request guarantor details".
   This is a process step, not a decision on the loan."""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

BRIEFS = {}
for letter_id in LETTERS:
    BRIEFS[letter_id] = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=LETTERS[letter_id],
            extracted_json=json.dumps(EXTRACTED[letter_id], indent=2),
        ),
        system_prompt=BRIEF_SYSTEM,
        temperature=0,
        max_tokens=800,
    )

print("L001 — Akosua Mensah\n")
print(BRIEFS["L001"])


L001 — Akosua Mensah

## Step 1: Strengths
* The applicant has 12 years of experience selling provisions at Makola Market.
* The applicant has a stable monthly profit of GHS 900.
* The applicant has saved GHS 2,500 with the susu scheme over the past two years without missing a contribution.
* The applicant has a guarantor, her sister, who is a teacher.

## Step 2: Risks / red flags
* The applicant's plan to expand into frozen foods is unverified and may pose a risk if not well-researched.
* The repayment plan relies on the applicant's ability to maintain or increase her current monthly profit, which is uncertain.
* The applicant's claim of making a monthly profit of GHS 900 is not verified.
* The guarantor's ability to fulfill her obligations is not assessed.

## Step 3: Missing information
* Detailed business plan for expanding into frozen foods.
* Verification of the applicant's monthly profit.
* Assessment of the guarantor's financial capability.
* Information about the applicant's 

In [32]:
print("L002 — Kwame Boateng\n")
print(BRIEFS["L002"])

L002 — Kwame Boateng

## Step 1: Strengths
* The applicant has a clear purpose for the loan, which is to repair his trotro engine and settle personal debts.
* The applicant is expecting an improvement in business after the festive season, indicating potential for increased income.

## Step 2: Risks / red flags
* The applicant states that business has been slow, which could impact his ability to repay the loan.
* The applicant does not have collateral, which increases the risk for the lender.
* The applicant's statement that he can pay back "whenever the money comes" is vague and does not provide a clear repayment plan.
* The applicant's reliance on the phrase "God willing everything will be fine" introduces uncertainty and lack of concrete planning.
* The claim that business will pick up after the festive season is an unverified expectation.

## Step 3: Missing information
* The applicant's current monthly profit or income.
* A detailed breakdown of the costs for repairing the trotro e

In [33]:
print("L006 — Kofi\n")
print(BRIEFS["L006"])

L006 — Kofi

## Step 1: Strengths
* The applicant, Kofi, is 22 and describes himself as "full of energy", which could be an advantage in managing and growing a business.
* Kofi's friends consider him to be "very business minded", suggesting he may have some innate ability or potential for entrepreneurship.

## Step 2: Risks / red flags
* The applicant has no prior experience in any of the businesses he wants to start, as indicated by "I have not started any of these yet".
* The lack of collateral or a guarantor increases the risk for the lender.
* The plan to repay the loan "when the businesses are booming" is based on future success without a clear plan or timeline, making it uncertain.
* The claim of being "trustworthy" is unverified and does not provide a tangible security for the loan.
* The proposal to start multiple businesses simultaneously (car washing, provision shop, and importing phones) could be overly ambitious and increase the risk of failure.

## Step 3: Missing informat

In [34]:
print("L003 — Efua Darko\n")
print(BRIEFS["L003"])

L003 — Efua Darko

## Step 1: Strengths
* The applicant has a registered business with a clear purpose for the loan.
* The applicant has a history of revenue, with a notable amount in December.
* The applicant has a fixed deposit that can be pledged as collateral.
* The applicant has provided sales records for the past 18 months.

## Step 2: Risks / red flags
* The applicant's claim of monthly profit averages GHS 2,800 is unverified.
* The reliance on the Christmas season for increased sales is a risk, as it is based on future events.
* The letter lacks detailed information about the business's current financial situation.
* The proposed repayment plan is based on projected income, which may not materialize.

## Step 3: Missing information
* Current financial statements, including balance sheet and income statement.
* Detailed breakdown of the costs of the industrial sewing machines and fabric stock.
* Information about the applicant's experience in managing larger amounts of capital.


**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** 1. L003 vs L006. Risks and Missing information worked well on both. For L006 the brief correctly flagged that Kofi has started none of the three businesses, that "business-minded" rests on friends' opinions, that there is no security, and that repayment is "based on hope rather than a concrete plan." For L003 it picked up the registered business, the pledgeable deposit and the sales records.

Strengths is where it failed, structurally. For L006 it lists "The applicant has a clear idea of the businesses they want to start," which is contradicted three lines later by its own Risks section: "the purpose of the loan is vague, with three different business ideas." Both cannot be true. The same pattern appears in L002, where an expected improvement after the festive season is a strength in one section and "an unverified expectation" in the next. The cause is my prompt: I required four sections, so the model filled all four whether or not grounded content existed. The fix is to permit emptiness, for example "If a section has no grounded content, write 'None identified from the letter.'"

A larger flaw appeared once I had all six briefs. Every application received the same next step, "Request supporting documents." L003, with eighteen months of records and a GHS 5,000 deposit, was routed identically to L006, which has nothing. The constraint that blocked approve and reject also collapsed the one field meant to differentiate cases.

2. Why forbid approve and reject. Practically, the model can verify nothing: not Akosua's GHS 900, not Efua's registration number, not the uncle's taxi. A verdict computed from unverified self-report is a fluent restatement of the applicant's own claims, and its polish would make it more persuasive to a busy officer than its evidence warrants. Ethically, a credit decision changes a person's life and someone must be accountable. If the model issues verdicts, accountability disperses (officer defers to system, institution points at vendor) and the applicant has no one to appeal to.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 
git status
git add .
git commit -m "prompts.py file created"
git push

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
def values_match(field, extracted, gold):
    # both missing counts as correct
    if extracted is None and gold is None:
        return True
    if extracted is None or gold is None:
        return False

    if field == "applicant_name":
        return extracted.strip().lower() == gold.strip().lower()

    if field == "purpose":
        # free text — token overlap, since wording will never match exactly
        e = set(extracted.lower().replace("/", " ").split())
        g = set(gold.lower().replace("/", " ").split())
        return len(e & g) / len(g) >= 0.5

    if field == "has_collateral_or_guarantor":
        return bool(extracted) == bool(gold)

    return float(extracted) == float(gold)  # numbers, exact


FIELDS = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]

table = {}
for field in FIELDS:
    row = {}
    for letter_id in GOLD:
        row[letter_id] = values_match(
            field, EXTRACTED[letter_id].get(field), GOLD[letter_id][field]
        )
    row["accuracy"] = sum(row.values()) / len(GOLD)
    table[field] = row

acc_df = pd.DataFrame(table).T
acc_df

,L001,L003,L006,accuracy
applicant_name,True,True,True,1.0
amount_ghs,True,True,True,1.0
purpose,True,True,True,1.0
monthly_profit_ghs,True,True,True,1.0
has_collateral_or_guarantor,True,True,True,1.0
repayment_months,True,True,True,1.0


### Part 4.2 — Reliability: is the system consistent?

In [26]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

# Reliability: same letter, five runs at each temperature
for temp in [0.0, 1.0]:
    results = [extract_fields(LETTERS["L004"], temperature=temp) for _ in range(5)]

    valid = [r for r in results if r is not None]
    fingerprints = [json.dumps(r, sort_keys=True) for r in valid]
    unique = set(fingerprints)

    print(f"temperature = {temp}")
    print(f"  valid JSON:     {len(valid)}/5")
    print(f"  unique outputs: {len(unique)}")
    for fp in unique:
        print(f"    {fp}")
    print()

temperature = 0.0
  valid JSON:     5/5
  unique outputs: 1
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}

temperature = 1.0
  valid JSON:     5/5
  unique outputs: 2
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for my poultry farm at Nsawam for feed and 500 new layers", "repayment_months": 18}



### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

# ---------- Test 1: question about a detail not in the letter ----------
probe_1 = ask_llm(
    f"""Loan application letter:
\"\"\"{LETTERS["L001"]}\"\"\"

What is the applicant's credit score?""",
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0,
)
print("TEST 1 — credit score of L001 (not stated anywhere in the letter)")
print(probe_1)

# ---------- Test 2: irrelevant and empty input to the extractor ----------
WEATHER = """Accra will be partly cloudy today with a high of 31C and a
70% chance of afternoon showers. Winds southwest at 12 km/h. Humidity 84%.
Tomorrow: scattered thunderstorms, high of 29C."""

print("TEST 2a — weather report fed to the extractor")
print(json.dumps(extract_fields(WEATHER), indent=2))

print("\nTEST 2b — empty string fed to the extractor")
print(json.dumps(extract_fields(""), indent=2))

TEST 1 — credit score of L001 (not stated anywhere in the letter)
The loan application letter does not mention the applicant's credit score. It provides information about the applicant's business experience, loan request, monthly profit, savings, and proposed repayment plan, but does not include a credit score.
TEST 2a — weather report fed to the extractor
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}

TEST 2b — empty string fed to the extractor
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** 1. Accuracy. 100%, with all six fields correct on all three gold letters, 18 out of 18. No field failed, so I cannot name a hardest one from this data.

That deserves less weight than it looks. Eighteen data points cannot separate a reliable extractor from a lucky one, and the gold set excludes the ambiguous letters. I also had to make a scoring decision the lab left open: it fixes rules for names and numbers but not for purpose, which is free text. Mine returned "buy a deep freezer and expand into frozen foods" against gold "buy deep freezer / expand into frozen foods," which is semantically identical but not string identical, so I scored it by 50% token overlap.

The genuinely hard case is outside the gold set. For L004 the extractor returned monthly_profit_ghs: 1500 from "around GHS 1,500 in a good month," in a letter that also describes losing birds to flu. Defensible, but it flattens a hedged best case into a flat figure, and everything downstream (including my own brief, which receives the JSON) sees 1500 with no trace of "sometimes."

2. Reliability. Temperature 0: 5/5 valid JSON, 1 unique output. Temperature 1.0: 5/5 valid, 2 unique. Every number and boolean was identical across all ten runs, and only purpose varied.

Two conclusions. JSON validity is a near-useless health check, since all ten parsed, so a pipeline checking only "is it parseable" would have caught nothing while storing two different values for one applicant. And temperature degrades weakly-constrained fields first: my schema pins the numbers to one defensible answer, while purpose admits many valid phrasings. The variation I saw is benign, but the mechanism that produced a longer accurate paraphrase is the one that would produce an inaccurate one.

3. Hallucination. All three probes passed.

Test 1 (PASS). Asked for L001's credit score: "The loan application letter does not mention the applicant's credit score..." It named the absence rather than estimating from her savings record.

Test 2a (PASS). Weather report: all six fields null. It did not scrape 31 or 12 from the temperature and wind figures.

Test 2b (PASS). Empty string: all nulls, no copying of the few-shot example.

One result is worth recording despite passing. I specified has_collateral_or_guarantor as boolean, not nullable. The model returned null anyway. The judgment is right, since false would have been a fabricated claim, but downstream code written as if record.has_collateral: would treat None as falsy and silently turn "unknown" into "no collateral." The safeguard is a schema validator at the boundary, not trust that the model honours the spec.

I should not overclaim from three probes on one model at one temperature. I did not test contradictory input, adversarial instructions inside a letter, or ambiguous units.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** 1. Who is harmed. A letter is a writing sample before it is a business case. L002 and L006 are weak on substance, but they are also the least fluently written, and my briefs partly penalise them for that. The L002 brief lists "reliance on the phrase 'God willing everything will be fine'" as a risk, a register of speech that carries no information about repayment capacity.

So the harm falls on applicants whose businesses are sound but whose letters are not: traders with limited formal schooling, people writing in a second or third language, anyone whose security is real but informal, such as an uncle's taxi, a susu record, or a cooperative's joint liability. That is close to the population microfinance exists to serve, so the error is not randomly distributed. It also compounds, because applicants who are declined never generate the repayment record that would have corrected the assessment.

My own results show the mechanism. The extractor collapsed "around GHS 1,500 in a good month" into 1500, and the brief then reasoned over that number as fact. Structured extraction discards exactly the hedges a human officer would weigh.

2. Third-party API in another country. The letters contain names, locations, registration numbers, income and family details, all personal data under Ghana's Data Protection Act, 2012 (Act 843). Sending them to Groq transfers identifiable data outside Ghana to servers under another jurisdiction's law, without the applicants' consent, while the institution remains legally accountable.

Before deploying I would check: registration with the Data Protection Commission and whether the stated purpose covers third-party AI processing; the lawful basis, noting that consent required as a condition of applying for credit is not meaningful consent; the provider's retention, sub-processing and training-on-data terms; whether a data processing agreement with cross-border safeguards exists; and whether the same result is achievable with less exposure, by redacting identifiers before transmission or self-hosting an open model.

3. Two safeguards.

A mandatory human decision point. No adverse action on model output alone, and a named officer decides and signs. This needs teeth against automation bias, which is the real risk, because my briefs are fluent regardless of whether their content is sound, as L006's manufactured strengths show. Officers should see the original letter beside the brief, and declines should require a reason in the officer's own words.

Complete logging with periodic disparate-impact review. Store the input, the prompt version, the model version, the raw output and the final decision. Without it the institution cannot answer "why was this applicant flagged," cannot detect a silent model change, and cannot reproduce a decision under challenge, and as Part 1.2 showed, temperature 0 does not guarantee reproducibility. Then review quarterly whether outcomes correlate with letter quality, region or gender rather than with actual repayment.

In [ ]:
# Measure real token usage for the Section 5 cost estimate
def usage_for(user_prompt, system_prompt, max_tokens=800):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=max_tokens,
    )
    return r.usage


letter = LETTERS["L001"]

u_sum = usage_for(SUMMARY_PROMPT_V2.format(letter_text=letter), SUMMARY_SYSTEM_V2, 300)
u_ext = usage_for(EXTRACT_PROMPT.format(letter_text=letter), EXTRACT_SYSTEM, 300)
u_brf = usage_for(
    BRIEF_PROMPT.format(
        letter_text=letter, extracted_json=json.dumps(EXTRACTED["L001"], indent=2)
    ),
    BRIEF_SYSTEM,
    800,
)

for name, u in [("summary", u_sum), ("extraction", u_ext), ("brief", u_brf)]:
    print(
        f"{name:12} prompt={u.prompt_tokens:5}  completion={u.completion_tokens:5}  total={u.total_tokens:5}"
    )

per_app = u_sum.total_tokens + u_ext.total_tokens + u_brf.total_tokens
print(f"\nper application: {per_app} tokens")
print(f"1,000 applications/month: {per_app * 1000:,} tokens")

summary      prompt=  267  completion=   86  total=  353
extraction   prompt=  500  completion=   65  total=  565
brief        prompt=  482  completion=  261  total=  743

per application: 1661 tokens
1,000 applications/month: 1,661,000 tokens


---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** 1. Prompting vs hyperparameter tuning. Both are empirical loops that punish changing several things at once. But hyperparameters are numeric and ordered so search can be automated, whereas prompts are natural language with no gradient to follow, and the feedback is qualitative: judging whether a summary laundered a hedge meant reading output against source by hand, not reading off a validation loss. Prompts are also brittle in a way hyperparameters are not, because the artefact I tuned is not the model, and a provider can change the model beneath a fixed prompt and silently change my results.

2. Trust. No. The result that decides it is that all six applications received the same next step, "Request supporting documents," so L003 with eighteen months of records was routed identically to L006 with nothing. A brief that discriminates in its prose while recommending the same action regardless is worse than no recommendation, because the format implies a judgment was made. I would trust the extraction and summarization as a reviewed first pass, given 18/18 against gold, but not the judgment layer.

3. Cost and scale. Measured on L001, one application costs three calls totalling 1,661 tokens: summary 353, extraction 565 and brief 743. At that rate 1,000 applications per month is about 1.66 million tokens. The shape is as informative as the total, since extraction is the most prompt-heavy call (500 in for 65 out, because the schema and few-shot example are resent every time) and would be the obvious thing to trim at scale. This volume sits within or near several free tiers and would cost single-digit dollars at paid rates, so price should not decide the provider; what should decide it is where the data is processed and under what law, whether the provider trains on submitted data, and version stability, since a silent model swap would invalidate my evaluation without warning.

4. Why an API beats training here. I have no training data: six letters and three gold labels is not a dataset, and a real institution would need years of letters labelled with outcomes before it could train anything comparable. The foundation model supplies general language competence, such as reading hedged prose and emitting valid JSON, that no institution could learn from its own loan book, and it took hours rather than months. It stops being the right choice when data cannot leave the jurisdiction, when volume makes per-token cost dominant, or when the task is narrow and well-labelled enough that a small classifier would be more accurate and easier to audit.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.